#  Tiny LLM on NASA Space Data  

## *Introduction*  
In this notebook, we will train a **very small Language Model (LLM)** on **NASA space-related datasets**.  
The goal is educational – to understand the basic steps of preparing data, building a small model, training it, and generating text.  

We will use:  
* **eva.csv** → Extravehicular activities (spacewalks)  
* **neo.csv / nearest-earth-objects.csv** → Near-Earth Objects data  
* **cleaned_5250.csv** → Cleaned dataset for training  

### *What we will do:*  
1. Load and explore the dataset  
2. Prepare text data for training  
3. Tokenize the text  
4. Define a very small LLM (RNN-based)  
5. Train the model  
6. Generate sample text about space  

---


#  Import Libraries and Setup

We will use **PyTorch** for building the model.  
The Kaggle notebook may run on CPU or GPU, so we will detect the device first.


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import random

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


#  Load and Explore Dataset  

We will start with **eva.csv** (Extravehicular Activities - spacewalks).  
Steps:  
* Read the CSV file  
* Display a few rows  
* Understand what columns exist (we will later convert them into text for training)  


In [19]:
import pandas as pd

data = {
    "prompt": [
        "Earth", "Mars", "Jupiter", "Saturn", "Venus", "Mercury", "Neptune", "Uranus",
        "Asteroid", "Comet", "Moon", "Sun", "Black Hole", "2020 XE8", "2011 HJ5"
    ],
    "response": [
        "Earth is the third planet from the Sun and the only one known to support life.",
        "Mars is a cold desert world often called the Red Planet because of its reddish soil.",
        "Jupiter is the largest planet in our solar system and has a giant storm called the Great Red Spot.",
        "Saturn is famous for its bright rings made of ice and rock.",
        "Venus is covered by thick clouds and has a very hot surface, hotter than Mercury.",
        "Mercury is the smallest planet and the closest one to the Sun.",
        "Neptune is a blue planet with strong winds and storms, located far from the Sun.",
        "Uranus rotates on its side and appears blue because of methane in its atmosphere.",
        "Asteroids are rocky objects that orbit the Sun, usually between Mars and Jupiter.",
        "Comets are icy objects that release gas and dust, creating bright tails when near the Sun.",
        "The Moon is Earth’s natural satellite and affects our tides.",
        "The Sun is a star at the center of the solar system that provides heat and light to Earth.",
        "A black hole is a region of space where gravity is so strong that not even light can escape.",
        "2020 XE8 is a near-Earth asteroid that orbits the Sun.",
        "2011 HJ5 is a small asteroid discovered in 2011 that passes near Earth’s orbit."
    ]
}

df = pd.DataFrame(data)
df.to_csv("/kaggle/working/space_text_dataset.csv", index=False)
print(df.head())


    prompt                                           response
0    Earth  Earth is the third planet from the Sun and the...
1     Mars  Mars is a cold desert world often called the R...
2  Jupiter  Jupiter is the largest planet in our solar sys...
3   Saturn  Saturn is famous for its bright rings made of ...
4    Venus  Venus is covered by thick clouds and has a ver...


In [20]:
df.to_csv("/kaggle/working/space_text_dataset.csv", index=False)


In [21]:
# Load eva.csv file
eva_df = pd.read_csv("/kaggle/input/nasa-training/eva.csv")

# Show first few rows
eva_df.head()


,EVA #,Country,Crew,Vehicle,Date,Duration,Purpose
0,1.0,USA,Ed White,Gemini IV,06/03/1965,0:36,First U.S. EVA. Used HHMU and took photos. G...
1,2.0,USA,David Scott,Gemini VIII,"March 16-17, 1966",0:00,HHMU EVA cancelled before starting by stuck on...
2,3.0,USA,Eugene Cernan,Gemini IX-A,06/05/1966,2:07,"Inadequate restraints, stiff 25ft umbilical an..."
3,4.0,USA,Mike Collins,Gemini X,07/19/1966,0:50,Standup EVA. UV photos of stars. Ended by ey...
4,5.0,USA,Mike Collins,Gemini X,07/20/1966,0:39,Retrieved MMOD experiment from docked Agena. ...


#  Merge All CSVs into a Single Text Corpus  

We have multiple datasets:  
* eva.csv → Extravehicular Activities (spacewalks)  
* neo.csv → Near-Earth Objects  
* nearest-earth-objects.csv → Historical NEO data  
* cleaned_5250.csv & neo_v2.csv → Cleaned datasets  

We will:  
1. Load each CSV file.  
2. Convert each row into **string text**.  
3. Merge everything into **one big text corpus** for training our LLM.  


In [23]:
# List of all CSV files (NASA + natural language)
csv_files = [
    "/kaggle/input/nasa-training/eva.csv",
    "/kaggle/input/nasa-training/neo.csv",
    "/kaggle/input/nasa-training/nearest-earth-objects(1910-2024).csv",
    "/kaggle/input/nasa-training/cleaned_5250.csv",
    "/kaggle/input/nasa-training/neo_v2.csv",
    "/kaggle/working/space_text_dataset.csv"   # 👈 Your new natural-language dataset
]

# Load and merge text
text_corpus = ""

for file in csv_files:
    try:
        df = pd.read_csv(file)
        # Convert entire dataframe to string text
        text_corpus += df.astype(str).apply(" ".join, axis=1).str.cat(sep=" ") + " "
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Show preview
print(text_corpus[:500])
print("\nCorpus length:", len(text_corpus))


1.0 USA Ed White Gemini IV 06/03/1965 0:36 First U.S. EVA. Used HHMU and took  photos.  Gas flow cooling of 25ft umbilical overwhelmed by vehicle ingress work and helmet fogged.  Lost overglove.  Jettisoned thermal gloves and helmet sun visor 2.0 USA David Scott Gemini VIII March 16-17, 1966 0:00 HHMU EVA cancelled before starting by stuck on vehicle thruster that ended mission early 3.0 USA Eugene Cernan Gemini IX-A 06/05/1966 2:07 Inadequate restraints, stiff 25ft umbilical and high workloads 

Corpus length: 53111032


#  Character-Level Tokenization  

We will:  
1. Get all unique characters from the text corpus.  
2. Assign each character an **integer ID**.  
3. Convert the text corpus into a list of integers (tokens).  


In [24]:
# Get unique characters
chars = sorted(list(set(text_corpus)))
vocab_size = len(chars)

# Create mappings
stoi = {ch: i for i, ch in enumerate(chars)}  # char → int
itos = {i: ch for i, ch in enumerate(chars)}  # int → char

# Encode (text → integers)
def encode(text):
    return [stoi[ch] for ch in text]

# Decode (integers → text)
def decode(tokens):
    return "".join([itos[i] for i in tokens])

# Convert corpus into tokens
tokens = encode(text_corpus)

print("Vocabulary size:", vocab_size)
print("Sample tokens:", tokens[:50])
print("Decoded back:", decode(tokens[:50]))


Vocabulary size: 84
Sample tokens: [13, 10, 12, 0, 44, 42, 24, 0, 28, 54, 0, 46, 58, 59, 70, 55, 0, 30, 55, 63, 59, 64, 59, 0, 32, 45, 0, 12, 18, 11, 12, 15, 11, 13, 21, 18, 17, 0, 12, 22, 15, 18, 0, 29, 59, 68, 69, 70, 0, 44]
Decoded back: 1.0 USA Ed White Gemini IV 06/03/1965 0:36 First U


#  Create Dataset and DataLoader

* We will create fixed-length context windows (`block_size`) from the token sequence.
* Each training sample: input = tokens[i : i+block_size], target = tokens[i+1 : i+1+block_size]
* We'll use a PyTorch Dataset + DataLoader for batching.


In [27]:
import torch
from torch.utils.data import Dataset, DataLoader

# hyperparams for dataset
block_size = 64   # context length (small, keeps model light)
batch_size = 32   # batch size

# Convert tokens (a python list) into a torch tensor
tokens_tensor = torch.tensor(tokens, dtype=torch.long)

class CharDataset(Dataset):
    def __init__(self, data, block_size):
        # data: 1D LongTensor of token ids
        self.data = data
        self.block_size = block_size
    def __len__(self):
        # number of training windows
        return max(0, (len(self.data) - 1) // self.block_size)
    def __getitem__(self, idx):
        # create windowed samples using idx * block_size as a simple stride
        start = idx * self.block_size
        x = self.data[start : start + self.block_size]
        y = self.data[start + 1 : start + 1 + self.block_size]
        # if last window shorter than block_size, pad (very unlikely here)
        if x.size(0) < self.block_size:
            pad_len = self.block_size - x.size(0)
            x = torch.cat([x, torch.zeros(pad_len, dtype=torch.long)])
            y = torch.cat([y, torch.zeros(pad_len, dtype=torch.long)])
        return x, y

dataset = CharDataset(tokens_tensor, block_size)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

print("Dataset samples:", len(dataset))
batch_x, batch_y = next(iter(dataloader))
print("Batch shapes:", batch_x.shape, batch_y.shape)  # (B, T)


Dataset samples: 829859
Batch shapes: torch.Size([32, 64]) torch.Size([32, 64])


#  Tiny Transformer LM

* Architecture components:
  * token embedding + positional embeddings
  * small `TransformerEncoder` stack with causal masking
  * linear head to predict next token logits
* Keep sizes small for quick training on Kaggle.


In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

d_model = 128    # embedding dimension (small)
nhead = 4        # number of attention heads
n_layers = 2     # encoder layers
dim_feedforward = 256  # FFN hidden dimension
dropout = 0.1

def generate_causal_mask(sz, device):
    # square subsequent mask for causal self-attention (True -> masked)
    mask = torch.triu(torch.ones(sz, sz, device=device) * float('-inf'), diagonal=1)
    return mask  # shape (T, T), use directly as src_mask for nn.Transformer

class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=128, nhead=4, n_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='relu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        # idx: (B, T)
        B, T = idx.shape
        assert T <= self.block_size, "T must be <= block_size"
        positions = torch.arange(T, device=idx.device).unsqueeze(0).expand(B, T)  # (B, T)
        x = self.tok_emb(idx) + self.pos_emb(positions)  # (B, T, E)
        # TransformerEncoder expects (T, B, E)
        x = x.transpose(0, 1)  # (T, B, E)
        src_mask = generate_causal_mask(T, idx.device)  # (T, T) with -inf above diag
        x = self.transformer(x, mask=src_mask)  # (T, B, E)
        x = x.transpose(0, 1)  # (B, T, E)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0):
        # idx: (B, T) initial context (on device)
        for _ in range(max_new_tokens):
            t = idx.shape[1]
            start = max(0, t - self.block_size)
            cur = idx[:, start:]  # keep only last block_size tokens
            logits = self.forward(cur)  # (B, Tcur, V)
            logits = logits[:, -1, :] / temperature  # last token logits (B, V)
            probs = F.softmax(logits, dim=-1)  # (B, V)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# instantiate
model = TinyTransformerLM(vocab_size=vocab_size, block_size=block_size, d_model=d_model, nhead=nhead, n_layers=n_layers, dim_feedforward=dim_feedforward, dropout=dropout).to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))


#  Training Loop

* Use CrossEntropyLoss (applies softmax + log-loss).
* Keep epochs small (e.g., `epochs = 5`) for demonstration; adjust as needed.
* Print loss every few steps.


In [ ]:
import torch.optim as optim
from tqdm import tqdm

epochs = 5
lr = 3e-4

optimizer = optim.AdamW(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(1, epochs + 1):
    running_loss = 0.0
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch}")
    for i, (x, y) in pbar:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)  # (B, T, V)
        # reshape for loss: (B*T, V) and (B*T)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i + 1) % 10 == 0:
            pbar.set_postfix(loss=running_loss / (i + 1))
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch} finished — avg loss: {avg_loss:.4f}")


In [ ]:
import torch
import torch.nn as nn
torch.save(model.state_dict(), "my_model.pth")



In [ ]:
# Reload trained model later
model.load_state_dict(torch.load("/kaggle/working/my_model.pth"))
model.eval()


#  Generate text from the trained model

* Provide a short prompt (string), encode it, and let the model autoregressively generate new tokens.
* We'll decode tokens back to characters for readability.


In [ ]:
model.eval()

def generate_from_prompt(prompt, max_new_tokens=200, temperature=1.0):
    # encode prompt
    prompt_tokens = torch.tensor([encode(prompt)], dtype=torch.long).to(device)  # (1, T)
    out = model.generate(prompt_tokens, max_new_tokens=max_new_tokens, temperature=temperature)  # (1, T+gen)
    out_list = out[0].tolist()
    return decode(out_list)

# Example prompts (choose any)
prompt = "Jupiter"
generated = generate_from_prompt(prompt, max_new_tokens=300, temperature=0.9)
print("----- GENERATED TEXT -----\n")
print(generated)
